In [1]:
#!pip install crewai==0.28.8 crewai-tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of crewai-tools to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of crewai-tools to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of embedchain to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of embedchain to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the depe

In [2]:
#!pip install -q google-generativeai

In [3]:
# For Scraping
#!pip install -qU tavily-python scrapegraph-py

In [4]:
# !pip install -qU 'crewai==0.30.11[tools]'
# !pip install -qU agentops
# !pip install -qU tavily-python
# !pip install -qU scrapegraph-py

In [5]:
#!pip install -qU crewai crewai-tools agentops

In [6]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource
import agentops
from google.colab import userdata
from pydantic import BaseModel, Field
from typing import List
from tavily import TavilyClient
from scrapegraph_py import Client

import os
import json

AttributeError: `np.float_` was removed in the NumPy 2.0 release. Use `np.float64` instead.

In [ ]:
from google.colab import userdata

print(userdata.get("GeminiAPIKey"))

https://app.agentops.ai/get-started >>> agentops

https://dashboard.scrapegraphai.com/  >>>> scrapegraphai

https://app.tavily.com/home?ph_referrer=gemini.google.com >>> tavily


In [ ]:
# API KEYS
# =============================
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
TAVILY_API_KEY = userdata.get("tvly-search")
SCRAPE_API_KEY = userdata.get("scrapegraph")

assert OPENAI_API_KEY, "Missing OpenAI API Key"
assert TAVILY_API_KEY, "Missing Tavily API Key"
assert SCRAPE_API_KEY, "Missing ScrapeGraph API Key"

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
basic_llm = LLM(model="gpt-4o-mini")

search_client = TavilyClient(api_key=TAVILY_API_KEY)
scrape_client = Client(api_key=SCRAPE_API_KEY)

In [ ]:
about_company = "Rowai is a company that provides AI solutions to improve product search and recommendations."

company_context = StringKnowledgeSource(content=about_company)

In [ ]:
# import os
# os.environ["OPENAI_API_KEY"] = userdata.get('GeminiAPIKey')

In [ ]:
# os.environ["AGENTOPS_API_KEY"] = userdata.get('agentops-colab')

# agentops.init(
#     api_key=userdata.get('agentops-colab'),
#     skip_auto_end_session=True,
#     default_tags=['crewai']
# )

In [ ]:
# print(agentops.get_client().config.exporter_endpoint)

In [ ]:
# output_dir = "./ai-agent-output"
# os.makedirs(output_dir, exist_ok=True)

# basic_llm = LLM(
#     model="gemini-pro", # Changed from "gemini/gemini-1.5-flash-latest" to "gemini-pro"
#     api_key=userdata.get('GeminiAPIKey')
# )

# search_client = TavilyClient(api_key=userdata.get('tvly-search'))
# scrape_client = Client(api_key=userdata.get('scrapegraph'))

In [ ]:
# no_keywords = 10

# about_company = "Rowai is a company that provides AI solutions to help websites refine their search and recommendation systems."

# company_context = StringKnowledgeSource(
#     content=about_company
# )

# ***Setup Agents***

# ***Agent1 Planner***
for searching


In [ ]:
class SuggestedSearchQueries(BaseModel):
    queries: List[str]

search_agent = Agent(
    role="Search Query Generator",
    goal="Generate specific product search queries",
    backstory="Expert in ecommerce search optimization",
    llm=basic_llm,
    verbose=True,
)

search_task = Task(
    description="""
    Generate 10 specific search queries for buying a Tablet in Egypt.
    Focus on ecommerce product pages (not blogs).
    Include brands like Samsung, Apple, Lenovo.
    """,
    expected_output="JSON with list of queries",
    output_json=SuggestedSearchQueries,
    agent=search_agent
)

In [ ]:
# class SuggestedSearchQueries(BaseModel):
#     # This defines a list of strings called 'queries'
#     # 'Field' adds constraints: it must have at least 1 item and no more than 'no_keywords'
#     queries: List[str] = Field(..., title="Suggested search queries to be passed to the search engine",
#                                min_items=1, max_items=no_keywords)

# search_queries_recommendation_agent = Agent(
#     role="Search Queries Recommendation Agent",
#     # The 'goal' defines exactly what this agent is trying to achieve.
#     goal="\n".join([
#                 "To provide a list of suggested search queries to be passed to the search engine.",
#                 "The queries must be varied and looking for specific items."
#             ]),
#     # The 'backstory' gives the AI context, helping it perform more professionally.
#     backstory="The agent is designed to help in looking for products by providing a list of suggested search queries to be passed to the search engine based on the context provided.",
#     # 'llm' tells the agent which AI model to use (Gemini in your case).
#     llm=basic_llm,
#     # 'verbose=True' allows you to see the agent's internal "thinking" process in the console.
#     verbose=True,
# )

# search_queries_recommendation_task = Task(
#     # 'description' is the actual prompt sent to the AI.
#     # Notice the placeholders in {curly_brackets} which will be filled when we run the crew.
#     description="\n".join([
#         "Rankyx is looking to buy {product_name} at the best prices (value for a price strategy)",
#         "The company targets any of these websites to buy from: {websites_list}",
#         "The company wants to reach all available products on the internet to be compared later in another stage.",
#         "The stores must sell the product in {country_name}",
#         "Generate at maximum {no_keywords} queries.",
#         "The search keywords must be in {language} language.",
#         "Search keywords must contain specific brands, types or technologies. Avoid general keywords.",
#         "The search query must reach an ecommerce webpage for product, and not a blog or listing page."
#     ]),
#     # 'expected_output' defines what a "successful" result looks like.
#     expected_output="A JSON object containing a list of suggested search queries.",
#     # 'output_json' forces the result into the Pydantic class we created in Step 1.
#     output_json=SuggestedSearchQueries,
#     # 'output_file' saves the final result to your Colab folder automatically.
#     output_file=os.path.join(output_dir, "step_1_suggested_search_queries.json"),
#     # 'agent' tells CrewAI which specialist is responsible for this job.
#     agent=search_queries_recommendation_agent
# )

# ***Agent2  Searcher***

In [ ]:
class SingleSearchResult(BaseModel):
    title: str
    url: str
    content: str
    score: float

class AllSearchResults(BaseModel):
    results: List[SingleSearchResult]

@tool
def search_tool(query: str):
    """Searches for information using TavilyClient."""
    try:
        res = search_client.search(query)
        return res.get("results", [])
    except:
        return []

search_engine_agent = Agent(
    role="Search Agent",
    goal="Find product links",
    backstory="An expert search agent focused on finding relevant product pages from various e-commerce websites.",
    llm=basic_llm,
    tools=[search_tool],
    verbose=True
)

search_engine_task = Task(
    description="Search and return product links only (no blogs).",
    expected_output="JSON of search results",
    output_json=AllSearchResults,
    agent=search_engine_agent
)

In [ ]:
# class SignleSearchResult(BaseModel):
#     title: str              # The headline of the search result
#     url: str                # The actual link to the product page
#     content: str            # A snippet of text from the page
#     score: float            # How relevant the search engine thinks this result is
#     search_query: str       # Which keyword was used to find this specific link

# class AllSearchResults(BaseModel):
#     # This groups multiple 'SignleSearchResult' objects into one list
#     results: List[SignleSearchResult]

# @tool
# def search_engine_tool(query: str):
#     """Useful for search-based queries. Use this to find current information..."""
#     # This calls the Tavily API to perform the actual Google-like search
#     return search_client.search(query)

# search_engine_agent = Agent(
#     role="Search Engine Agent",
#     goal="To search for products based on the suggested search query",
#     backstory="The agent is designed to help in looking for products by searching...",
#     llm=basic_llm,      # Using your Gemini model
#     verbose=True,       # Shows you the URLs it finds in real-time
#     tools=[search_engine_tool] # We 'hand' the search tool to this specific agent
# )

# search_engine_task = Task(
#     description="\n".join([
#         "The task is to search for products based on the suggested search queries.",
#         "You have to collect results from multiple search queries.",
#         # QUALITY CONTROL: Tell the agent to avoid blogs or spammy links
#         "Ignore any suspicious links or not an ecommerce single product website link.",
#         # THRESHOLD: Only keep results that are highly relevant (score_th)
#         "Ignore any search results with confidence score less than ({score_th}).",
#         "The search results will be used to compare prices...",
#     ]),
#     expected_output="A JSON object containing the search results.",
#     output_json=AllSearchResults, # Validates that we get a list of URLs
#     output_file=os.path.join(output_dir, "step_2_search_results.json"), # Saves the links
#     agent=search_engine_agent
# )

# ***Agent3  Scraper***

In [ ]:
class ProductSpec(BaseModel):
    specification_name: str
    specification_value: str

class Product(BaseModel):
    product_title: str
    product_url: str
    product_current_price: float
    product_specs: List[ProductSpec]

class AllProducts(BaseModel):
    products: List[Product]

@tool
def scrape_tool(url: str):
    """Extracts product details from a given URL using scrape_client."""
    try:
        data = scrape_client.smartscraper(
            website_url=url,
            user_prompt="Extract product title, price, and specs"
        )
        return data
    except:
        return {"error": "failed"}

scraping_agent = Agent(
    role="Scraper",
    goal="Extract product data",
    backstory="An expert at extracting specific product details from various e-commerce websites.",
    llm=basic_llm,
    tools=[scrape_tool],
    verbose=True
)

scraping_task = Task(
    description="Extract product details from links.",
    expected_output="JSON of products",
    output_json=AllProducts,
    agent=scraping_agent
)

In [ ]:
# class ProductSpec(BaseModel):
#     specification_name: str
#     specification_value: str

# class SingleExtractedProduct(BaseModel):
#     page_url: str = Field(..., title="The original url of the product page")
#     product_title: str = Field(..., title="The title of the product")
#     product_image_url: str = Field(..., title="The url of the product image")
#     product_url: str = Field(..., title="The url of the product")
#     product_current_price: float = Field(..., title="The current price of the product")
#     product_original_price: float = Field(title="The original price of the product before discount. Set to None if no discount", default=None)
#     product_discount_percentage: float = Field(title="The discount percentage of the product. Set to None if no discount", default=None)

#     product_specs: List[ProductSpec] = Field(..., title="The specifications of the product. Focus on the most important specs to compare.", min_items=1, max_items=5)

#     agent_recommendation_rank: int = Field(..., title="The rank of the product to be considered in the final procurement report. (out of 5, Higher is Better) in the recommendation list ordering from the best to the worst")
#     agent_recommendation_notes: List[str]  = Field(..., title="A set of notes why would you recommend or not recommend this product to the company, compared to other products.")


# class AllExtractedProducts(BaseModel):
#     products: List[SingleExtractedProduct]


# @tool
# def web_scraping_tool(page_url: str):
#     """
#     An AI Tool to help an agent to scrape a web page

#     Example:
#     web_scraping_tool(
#         page_url="https://www.noon.com/egypt-en/15-bar-fully-automatic-espresso-machine-1-8-l-1500"
#     )
#     """
#     details = scrape_client.smartscraper(
#         website_url=page_url,
#         user_prompt="Extract ```json\n" + SingleExtractedProduct.schema_json() + "```\n From the web page"
#     )

#     return {
#         "page_url": page_url,
#         "details": details
#     }

# scraping_agent = Agent(
#     role="Web scraping agent",
#     goal="To extract details from any website",
#     backstory="The agent is designed to help in looking for required values from any website url. These details will be used to decide which best product to buy.",
#     llm=basic_llm,
#     tools=[web_scraping_tool],
#     verbose=True,
# )

# scraping_task = Task(
#     description="\n".join([
#         "The task is to extract product details from any ecommerce store page url.",
#         "The task has to collect results from multiple pages urls.",
#         "Collect the best {top_recommendations_no} products from the search results.",
#     ]),
#     expected_output="A JSON object containing products details",
#     output_json=AllExtractedProducts,
#     output_file=os.path.join(output_dir, "step_3_search_results.json"),
#     agent=scraping_agent
# )

# ***Agent4  Reporter***

In [ ]:
report_agent = Agent(
    role="Report Generator",
    goal="Create HTML report",
    backstory="An expert in generating professional and detailed HTML reports, skilled in using frameworks like Bootstrap to present information clearly and aesthetically.",
    llm=basic_llm,
    verbose=True
)

report_task = Task(
    description="""
    Create a professional HTML report comparing products and prices.
    Use Bootstrap.
    """,
    expected_output="HTML report",
    output_file="report.html",
    agent=report_agent
)

In [ ]:
# procurement_report_author_agent = Agent(
#     role="Procurement Report Author Agent",
#     goal="To generate a professional HTML page for the procurement report",
#     backstory="The agent is designed to assist in generating a professional HTML page for the procurement report after looking into a list of products.",
#     llm=basic_llm,
#     verbose=True,
# )

# procurement_report_author_task = Task(
#     description="\n".join([
#         "The task is to generate a professional HTML page for the procurement report.",
#         "You have to use Bootstrap CSS framework for a better UI.",
#         "Use the provided context about the company to make a specialized report.",
#         "The report will include the search results and prices of products from different websites.",
#         "The report should be structured with the following sections:",
#         "1. Executive Summary: A brief overview of the procurement process and key findings.",
#         "2. Introduction: An introduction to the purpose and scope of the report.",
#         "3. Methodology: A description of the methods used to gather and compare prices.",
#         "4. Findings: Detailed comparison of prices from different websites, including tables and charts.",
#         "5. Analysis: An analysis of the findings, highlighting any significant trends or observations.",
#         "6. Recommendations: Suggestions for procurement based on the analysis.",
#         "7. Conclusion: A summary of the report and final thoughts.",
#         "8. Appendices: Any additional information, such as raw data or supplementary materials.",
#     ]),

#     expected_output="A professional HTML page for the procurement report.",
#     output_file=os.path.join(output_dir, "step_4_procurement_report.html"),
#     agent=procurement_report_author_agent,
# )

# ***Run AI Crew***

In [ ]:
crew = Crew(
    agents=[
        search_agent,
        search_engine_agent,
        scraping_agent,
        report_agent
    ],
    tasks=[
        search_task,
        search_engine_task,
        scraping_task,
        report_task
    ],
    process=Process.sequential,
    knowledge_sources=[company_context]
)

In [ ]:
result = crew.kickoff()
print(result)

In [ ]:
import gradio as gr

def run_ai_agent(product_name):
    try:
        result = crew.kickoff(inputs={
            "product_name": product_name,
            "websites_list": ["amazon.eg", "jumia.com.eg", "noon.com"],
            "country_name": "Egypt",
            "no_keywords": 5,
            "language": "English",
            "score_th": 0.2,
            "top_recommendations_no": 5
        })
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"


interface = gr.Interface(
    fn=run_ai_agent,
    inputs=gr.Textbox(label="Enter Product (e.g., Tablet)"),
    outputs=gr.Textbox(label="Results"),
    title="🛒 AI Shopping Agent",
    description="Enter a product name and get best deals from different websites"
)

interface.launch()

In [ ]:
# crew_results = Rowai_crew.kickoff(
#     inputs={
#         "product_name": "Tablet",
#         "websites_list": ["www.amazon.eg", "www.jumia.com.eg", "www.noon.com/egypt-en"],
#         "country_name": "Egypt",
#         "no_keywords": 10,
#         "language": "English",
#         "score_th": 0.10,
#         "top_recommendations_no": 10
#     }
# )